# Optiver 方案二：时序与分组特征工程 + 随机森林

## 简介
以典型交易数据结构（如 stock_id、date_id 等）分组构造统计与归一化特征，采用随机森林进行 5 折交叉验证与全量拟合。

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error, r2_score, roc_auc_score, f1_score
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
np.random.seed(42)

## 数据加载

In [ ]:
DATA_DIRS = ["./data", "."]
TRAIN_FILES = ["train.csv", "optiver_train.csv"]
TEST_FILES = ["test.csv", "optiver_test.csv"]
def find_dataset_file(names):
    for d in DATA_DIRS:
        for n in names:
            p = os.path.join(d, n)
            if os.path.exists(p):
                return p
    return None
train_path = find_dataset_file(TRAIN_FILES)
test_path = find_dataset_file(TEST_FILES)
train_df = pd.read_csv(train_path) if train_path else None
test_df = pd.read_csv(test_path) if test_path else None
print("train_path", train_path)
print("test_path", test_path)
print(train_df.shape if train_df is not None else None)
print(test_df.shape if test_df is not None else None)

## 目标列与任务类型识别

In [ ]:
def detect_target(df):
    cols = df.columns.tolist()
    candidates = []
    for c in cols:
        cl = c.lower()
        if ("target" in cl) or ("movement" in cl) or (cl == "label") or (cl == "y"):
            candidates.append(c)
    if candidates:
        return candidates[0]
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    for c in numeric_cols[::-1]:
        if df[c].nunique() < len(df):
            return c
    return cols[-1]
def detect_task_type(y):
    uniq = pd.unique(y)
    if pd.api.types.is_numeric_dtype(y):
        ratio = len(uniq) / max(1, len(y))
        if len(uniq) <= 20 and ratio < 0.02:
            return "classification"
        return "regression"
    return "classification"
target_col = detect_target(train_df) if train_df is not None else None
task_type = detect_task_type(train_df[target_col]) if train_df is not None else None
print("target_col", target_col)
print("task_type", task_type)

## 分组与时序特征工程

In [ ]:
def make_features(df, target=None):
    if target is not None and target in df.columns:
        X = df.drop(columns=[target]).copy()
    else:
        X = df.copy()
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    if "stock_id" in X.columns:
        g = X.groupby("stock_id")
        for c in num_cols:
            m = g[c].transform("mean")
            X[c + "_stock_mean"] = m
            X[c + "_rel"] = X[c] / (m + 1e-9)
    day_col = None
    for dc in ["date", "date_id", "day"]:
        if dc in X.columns:
            day_col = dc
            break
    if day_col is not None:
        g = X.groupby(day_col)
        for c in num_cols:
            m = g[c].transform("mean")
            X[c + "_day_mean"] = m
            X[c + "_z"] = (X[c] - m)
    return X

## 交叉验证评估

In [ ]:
def run_cv(df, target, task):
    X = make_features(df, target)
    y = df[target]
    if task == "regression":
        model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
        cv = KFold(n_splits=5, shuffle=True, random_state=42)
        scores = []
        for tr, va in cv.split(X):
            Xtr, Xva = X.iloc[tr], X.iloc[va]
            ytr, yva = y.iloc[tr], y.iloc[va]
            model.fit(Xtr, ytr)
            p = model.predict(Xva)
            mse = mean_squared_error(yva, p)
            r2 = r2_score(yva, p)
            scores.append((mse, r2))
        return scores, model
    else:
        model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = []
        for tr, va in cv.split(X, y):
            Xtr, Xva = X.iloc[tr], X.iloc[va]
            ytr, yva = y.iloc[tr], y.iloc[va]
            model.fit(Xtr, ytr)
            proba = None
            try:
                proba = model.predict_proba(Xva)
            except Exception:
                proba = None
            if proba is not None and proba.shape[1] == 2:
                auc = roc_auc_score(yva, proba[:, 1])
                scores.append(auc)
            else:
                pred = model.predict(Xva)
                f1 = f1_score(yva, pred, average="macro")
                scores.append(f1)
        return scores, model
scores, model = run_cv(train_df, target_col, task_type) if train_df is not None else (None, None)
print("cv_scores", None if scores is None else scores[:3])

## 全量拟合与推断

In [ ]:
def fit_full_and_predict(train_df, test_df, target, task):
    if task == "regression":
        model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    else:
        model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    X_train = make_features(train_df, target)
    y_train = train_df[target]
    model.fit(X_train, y_train)
    if test_df is None:
        return None, model
    X_test = make_features(test_df, target=None)
    pred = model.predict(X_test)
    return pred, model
pred, model = fit_full_and_predict(train_df, test_df, target_col, task_type) if train_df is not None else (None, None)
if pred is not None:
    id_col = None
    if test_df is not None:
        for c in ["row_id", "id"]:
            if c in test_df.columns:
                id_col = c
                break
    sub = pd.DataFrame({id_col if id_col else "id": test_df[id_col] if id_col else np.arange(len(pred)), "prediction": pred})
    sub_path = "submission_scheme2.csv"
    sub.to_csv(sub_path, index=False)
    print(sub_path)
else:
    print(None)